# BERTScore on Colab and Merge with Local BLEU/ROUGE

Notebook ini menghitung BERTScore untuk file baseline BLEU/ROUGE yang sudah dibuat lokal, lalu menyimpan output yang siap di-merge di lokal.

In [ ]:
!pip -q install bert-score pandas torch

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from bert_score import score as bert_score

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Input/Output Paths

Cara pakai paling mudah:
1. Upload file `100_Pantun_Eksperimen_1200_with_bleu_rouge.csv` ke Colab (menu Files).
2. Sesuaikan `INPUT_CSV` dan `OUTPUT_BERT_CSV` di bawah.
3. Jalankan semua sel sampai selesai.
4. Download `OUTPUT_BERT_CSV` lalu merge di lokal pakai script Python.

In [ ]:
INPUT_CSV = '/content/100_Pantun_Eksperimen_1200_with_bleu_rouge.csv'
OUTPUT_BERT_CSV = '/content/100_Pantun_Eksperimen_1200_bertscore_only.csv'

REFERENCE_COL = 'sampiran_asli'
CANDIDATE_COL = 'sampiran_generated'
MERGE_KEYS = ['no', 'id_data_asli', 'model', 'setting']

BERTSCORE_MODEL = 'bert-base-multilingual-cased'
BATCH_SIZE_GPU = 32
BATCH_SIZE_CPU = 8

In [ ]:
df = pd.read_csv(INPUT_CSV)
required = MERGE_KEYS + [REFERENCE_COL, CANDIDATE_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

refs = [' '.join(str(x or '').strip().split()) for x in df[REFERENCE_COL].fillna('')]
cands = [' '.join(str(x or '').strip().split()) for x in df[CANDIDATE_COL].fillna('')]
batch_size = BATCH_SIZE_GPU if torch.cuda.is_available() else BATCH_SIZE_CPU

print('rows:', len(df))
print('batch_size:', batch_size)

In [ ]:
p, r, f1 = bert_score(
    cands,
    refs,
    model_type=BERTSCORE_MODEL,
    lang='id',
    batch_size=batch_size,
    verbose=True,
    rescale_with_baseline=False,
)

bert_df = df[MERGE_KEYS].copy()
bert_df['bertscore_p'] = [float(x) for x in p]
bert_df['bertscore_r'] = [float(x) for x in r]
bert_df['bertscore_f1'] = [float(x) for x in f1]

bert_df.to_csv(OUTPUT_BERT_CSV, index=False)
print('saved:', OUTPUT_BERT_CSV)
bert_df.head()

## Merge Kembali di Lokal

Setelah file BERTScore diunduh dari Colab, jalankan ini di lokal:

python scripts/article2/calc_baseline_nlg_metrics.py \
  --input data/article2/100_Pantun_Eksperimen_1200_xrasa_final.csv \
  --output data/article2/100_Pantun_Eksperimen_1200_with_bleu_rouge.csv \
  --summary data/article2/table_main_baselines_1200.csv \
  --bertscore-input data/article2/100_Pantun_Eksperimen_1200_bertscore_only.csv \
  --output-merged data/article2/100_Pantun_Eksperimen_1200_with_baselines.csv \
  --summary-merged data/article2/table_main_baselines_1200_with_bertscore.csv